<a href="https://colab.research.google.com/github/awitz23/Diabetes_prediction/blob/main/Minimale_finale_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Dependencies

In [11]:
!pip install ucimlrepo
!pip install catboost
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from ucimlrepo import fetch_ucirepo

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, precision_score, recall_score

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.feature_selection import RFE
from catboost import CatBoostClassifier
import pandas as pd
import numpy as np

from sklearn.metrics import classification_report, confusion_matrix, f1_score
from imblearn.over_sampling import SVMSMOTE
from imblearn.pipeline import Pipeline
from imblearn.metrics import classification_report_imbalanced

from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
import numpy as np

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from scipy.stats import sem
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

In [2]:
# --- Fetch dataset ---
cdc_diabetes_health_indicators = fetch_ucirepo(id=891)
X = cdc_diabetes_health_indicators.data.features
Y = cdc_diabetes_health_indicators.data.targets
y = Y.values.ravel()

#Finale Pipeline

In [6]:
# --- Step 1: Define feature types ---
binary_features = [
    'HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke', 'HeartDiseaseorAttack',
    'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare',
    'NoDocbcCost', 'DiffWalk', 'Sex'
]

ordinal_features = ['GenHlth', 'Age', 'Education', 'Income']
ratio_features = ['BMI', 'MentHlth', 'PhysHlth']

In [7]:
# --- Step 2: Define transformers ---
scaler = StandardScaler()
ordinal_encoder = OrdinalEncoder()

# Preprocessing pipeline for models that need scaling
preprocessor = ColumnTransformer([
    ('ratio', scaler, ratio_features),
    ('ordinal', ordinal_encoder, ordinal_features)
], remainder='passthrough')  # binary features remain untouched


rus = RandomUnderSampler(random_state=42)
X_bal, y_bal = rus.fit_resample(X, Y)

sampler = SMOTE(random_state=42)
X_sampled, y_sampled = sampler.fit_resample(X, y)
X_train, X_test, y_train, y_test = train_test_split(X_sampled, y_sampled, test_size=0.2, stratify=y_sampled, random_state=42)


In [8]:
rf = RandomForestClassifier(max_depth=None, n_estimators=100)

In [9]:
rf.fit(X_train, y_train)

RandomForestClassifier()

In [12]:
y_pred = rf.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred, digits=3))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0      0.902     0.822     0.860     43667
           1      0.836     0.911     0.872     43667

    accuracy                          0.866     87334
   macro avg      0.869     0.866     0.866     87334
weighted avg      0.869     0.866     0.866     87334

Accuracy: 0.8661231593651957
F1 Score: 0.8718151120466605
Precision: 0.8362603848985172
